# SVR - Solar Wastage
Train and evaluate an SVR model using features_regression_ready.csv.

In [ ]:
# Import required libraries
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

In [ ]:
# Define file paths (relative to notebooks/training/)
DATA_PATH = Path('../../data/processed/features_regression_ready.csv')  # Input data
MODEL_PATH = Path('../../models/svr_pipeline.pkl')  # Output model
METRICS_PATH = Path('../../models/svr_metrics.json')  # Output metrics
RANDOM_STATE = 42  # For reproducibility

In [ ]:
# Load and prepare data
df = pd.read_csv(DATA_PATH)
target_col = 'wasted_energy_kwh'  # Target variable to predict

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Define column types for preprocessing
categorical_cols = ['household', 'season', 'district']
numerical_cols = [c for c in X.columns if c not in categorical_cols]
X.head()

In [ ]:
# Define preprocessing pipelines for different column types
# Categorical: impute missing values with most frequent, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Numerical: impute missing values with median, then standardize
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Combine transformers into a single preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

In [ ]:
# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
# Create Support Vector Regressor model (default RBF kernel)
model = SVR()

# Build pipeline: preprocessing -> model
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

# Train the model and make predictions
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [ ]:
# Calculate evaluation metrics
r2 = r2_score(y_test, y_pred)            # R-squared (coefficient of determination)
mae = mean_absolute_error(y_test, y_pred) # Mean Absolute Error
rmse = root_mean_squared_error(y_test, y_pred)  # Root Mean Squared Error

# Display results
print('SVR Results')
print('=' * 50)
print(f'R2:   {r2:.4f}')
print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')

In [ ]:
# Save metrics and trained model
metrics = {'R2': float(r2), 'MAE': float(mae), 'RMSE': float(rmse)}

# Create models directory if it doesn't exist
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save pipeline (includes preprocessor + model)
joblib.dump(pipeline, MODEL_PATH)

# Save metrics as JSON
with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved:', MODEL_PATH, METRICS_PATH)

## Plot Actual vs Predicted

In [ ]:
# Plot Actual vs Predicted values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='k', linewidths=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Wasted Energy (kWh)')
plt.ylabel('Predicted Wasted Energy (kWh)')
plt.title('SVR: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.savefig('../../results/svr_actual_vs_predicted.png', dpi=150)
plt.show()